# Notebook 2: Signals and Drive Exports

Loads the pre-materialized Base Stack from Stage 1 and compiles all multi-scale signals directly into unified **33-band GeoTIFFs** exported to Google Drive.

### Pipeline Process:
For each scale (5km to 100km) and basin (Congo, Amazon):
1. Loads the Stage 1 Base Stack asset (~463m WGS84).
2. Computes the **cross-sectional FRIP** and **23 annual FRIP** Spearman correlations in memory.
3. Applies topographic + forest masks to the GEDI signals and aggregates them to the target scale.
4. Aggregates environmental covariates to the target scale.
5. Concatenates all **33 bands** and exports directly to Google Drive.

### Output GeoTIFF Structure (33 bands total):
- **`frip`** (1 band): Cross-sectional Spearman correlation
- **`FRIP_2001` ... `FRIP_2023`** (23 bands): Annual Spearman correlations
- **`uoi`**, **`rh98`**, **`gedi_n`** (3 GEDI bands): Openness, height, footprint count
- **`elevation`**, **`slope`**, **`hnd`**, **`precip`**, **`clay`**, **`forest_fraction`** (6 covariate bands)

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# Asset paths (from NB1)
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study regions (must match NB1)
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

# Analysis parameters
SCALES = list(range(5000, 105000, 5000))  # 5km to 100km in 5km steps
YEARS = list(range(2001, 2024))

# Masking thresholds
FOREST_COVER_THRESHOLD = 0.95
MAX_ELEVATION = 1000
MAX_SLOPE = 10

print("\u2713 Configuration loaded.")
print(f"  Scales: {SCALES[0]/1000:.0f}km - {SCALES[-1]/1000:.0f}km ({len(SCALES)} scales)")
print(f"  Basins: {[b[0] for b in BASINS]}")

In [ ]:
# =============================================================================
# BLOCK 2: IN-MEMORY COMPUTATION LOGIC
# =============================================================================

def build_scale_stack(base, basin_name, scale):
    """Computes all signals and covariates in memory and stacks into 33 bands."""
    base_proj = base.projection()
    
    # -------------------------------------------------------------------------
    # 1. FRIP computation (Cross-sectional + Annual)
    # -------------------------------------------------------------------------
    # Apply forest cover mask
    frip_masked = base.updateMask(
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
    )
    
    # Cross-sectional FRIP
    frip_cross = frip_masked.select(['flood_freq', 'Npp_median']).reduceResolution(
        reducer=ee.Reducer.spearmansCorrelation(),
        maxPixels=65535
    ).reproject(crs='EPSG:4326', scale=scale).select('correlation').rename('frip')
    
    # Apply quality cover mask (legacy pattern: keep cells with >10% valid coverage)
    frip_cross = frip_cross.updateMask(frip_cross.mask().gt(0.1))
    
    # Annual FRIP
    def get_annual_corr(year_index):
        year_index = ee.Number(year_index)
        year = ee.Number(2001).add(year_index)
        npp_band = ee.String('NPP_').cat(year.format('%d'))
        
        corr = frip_masked.select([npp_band, 'flood_freq']).reduceResolution(
            reducer=ee.Reducer.spearmansCorrelation(),
            maxPixels=65535
        ).reproject(crs='EPSG:4326', scale=scale).select('correlation')
        
        return corr.updateMask(corr.mask().gt(0.1)).set('year', year)
    
    annual_list = ee.List.sequence(0, len(YEARS) - 1).map(get_annual_corr)
    frip_annual = ee.ImageCollection.fromImages(annual_list).toBands()
    band_names = [f'FRIP_{y}' for y in YEARS]
    frip_annual = frip_annual.rename(band_names)
    
    # -------------------------------------------------------------------------
    # 2. GEDI signals (Masked and Aggregated)
    # -------------------------------------------------------------------------
    # Apply forest AND topo masks
    gedi_masked = base.updateMask(
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
        .And(base.select('elevation').lt(MAX_ELEVATION))
        .And(base.select('slope').lt(MAX_SLOPE))
    )
    
    uoi_agg = gedi_masked.select('GEDI_UOI').setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).rename('uoi')
    
    rh98_agg = gedi_masked.select('GEDI_rh98').setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).rename('rh98')
    
    n_agg = gedi_masked.select('GEDI_N').setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.sum(), maxPixels=65535
    ).rename('gedi_n')
    
    # -------------------------------------------------------------------------
    # 3. Covariates (Aggregated)
    # -------------------------------------------------------------------------
    covariates = ['elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction']
    covs_agg = base.select(covariates).setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    )
    
    # -------------------------------------------------------------------------
    # 4. Concatenate into a single 33-band stack
    # -------------------------------------------------------------------------
    stack = ee.Image.cat([
        frip_cross,    # 1 band
        frip_annual,   # 23 bands
        uoi_agg,       # 1 band
        rh98_agg,      # 1 band
        n_agg,         # 1 band
        covs_agg       # 6 bands
    ]).toFloat()
    
    return stack

print("\u2713 In-memory computation logic loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Loading Stage 1 base stack asset...")
    try:
        base = ee.Image(f'{ASSET_ROOT}/BaseStack_Congo')
        print("  \u2713 BaseStack_Congo loaded successfully")
        
        print("\nRunning combined stack unit tests (using Congo at 50km)...")
        stack = build_scale_stack(base, 'Congo', 50000)
        bands = stack.bandNames().getInfo()
        
        # Test 1: Band count
        assert len(bands) == 33, f"Expected 33 bands, got {len(bands)}: {bands}"
        print("  [1/3] \u2713 Correct band count: 33 bands assembled")
        
        # Test 2: Required bands
        required = ['frip', 'FRIP_2001', 'FRIP_2023', 'uoi', 'rh98', 'gedi_n',
                    'elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction']
        missing = [b for b in required if b not in bands]
        assert not missing, f"Missing required bands: {missing}"
        print("  [2/3] \u2713 All required signals and covariates present")
        
        # Test 3: Projection WGS84
        proj_info = stack.projection().getInfo()
        assert proj_info['crs'] == 'EPSG:4326', f"Expected EPSG:4326, got: {proj_info['crs']}"
        print(f"  [3/3] \u2713 Geodetic reference standard: {proj_info['crs']}")
        
        print(f"\n{'='*60}")
        print("  \u2713 ALL TESTS PASSED SUCCESSFULLY!")
        print("  Ready to launch Drive exports.")
        print(f"{'='*60}")
        
    except Exception as e:
        print(f"  \u2717 Test skipped or failed: {e}")
        print("    (This is expected if your Stage 1 BaseStack has not finished exporting yet.)")

run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: EXPORT TO DRIVE
# =============================================================================

def export_all_datasets(dry_run=True):
    """Launches exports for all 20 scales and both basins directly to Google Drive.
    
    Produces 40 GeoTIFFs total (20 scales x 2 basins).
    Saves to folder: 'DefaunationSynthesis/AnalysisStack/'
    """
    tasks = []
    
    for basin_name, basin_geom in BASINS:
        base = ee.Image(f'{ASSET_ROOT}/BaseStack_{basin_name}')
        
        for scale in SCALES:
            stack = build_scale_stack(base, basin_name, scale)
            
            task = ee.batch.Export.image.toDrive(
                image=stack,
                description=f'analysis_stack_{scale}_{basin_name}',
                folder='DefaunationSynthesis/AnalysisStack',
                fileNamePrefix=f'analysis_stack_{scale}_{basin_name}',
                region=basin_geom,
                scale=scale,
                crs='EPSG:4326',
                maxPixels=1e13
            )
            tasks.append((task, f'analysis_stack_{scale}_{basin_name}'))
            
    print(f"\u2713 {len(tasks)} Drive export tasks configured:")
    print(f"  ({len(SCALES)} scales x {len(BASINS)} basins)")
    
    if dry_run:
        print("\nDRY RUN. Call export_all_datasets(dry_run=False) to launch.")
    else:
        for task, name in tasks:
            task.start()
            print(f"  \u2713 Started Drive export: {name}")
        print("\n\u2713 All 40 Drive exports started!")
        print("  Monitor at: https://code.earthengine.google.com/tasks")

export_all_datasets(dry_run=True)